# KAVACH mechanistic-interpretability lab (nnsight)
Run **after** `KAVACH_Interpretability_GPU.ipynb` (it leaves the trained checkpoint, windows and interpretability artefacts in `MyDrive/kavach/outputs/processed`).
Runtime: CPU or GPU both fine (this notebook is analysis, not training); High-RAM recommended.

**Order matters:** run the *self-check* (cell 5) before trusting any plot. It proves the nnsight activations equal the repo's torch-hook activations and that a no-op patch changes nothing.

| Section | Question it answers | Tool |
|---|---|---|
| A. Embedding space | Does z_t separate attack families / stages? | PCA / t-SNE, interactive plotly |
| B. Timeline | How do score, forecast, truth and predicted stage move over a capture? | model outputs |
| C. Layers | Where does the decision form? Which layers are similar? | tuned lens, CKA, probes |
| D. Causal patching | Which layer *and subspace* is responsible for a detection? | **nnsight** activation patching |
| E. Dimensions | Which of the 128 embedding dims matter? | **nnsight** mean-ablation |
| F. SAE | Are SAE features real, and do they *cause* the score? | **nnsight** ablation, steering, fidelity |
| G. One window | Which inputs and which forecast? | grad x input, Monte-Carlo rollout |
| H. Model quality | Calibration, stage confusion | |

In [ ]:
import os, sys, glob, json, time
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/kavach'
!rm -rf /content/work && mkdir -p /content/work && unzip -q {DRIVE}/kavach_backend.zip -d /content/work
%cd /content/work/backend
!mkdir -p data && rm -rf data/processed && cp -r {DRIVE}/outputs/processed data/processed
!ls data/processed data/processed/interpretability | head -40

In [ ]:
!pip -q install nnsight torch-geometric captum shap scikit-learn pandas pyyaml scapy sqlalchemy fastapi pytest matplotlib plotly
import nnsight, torch, torch_geometric
print('nnsight', getattr(nnsight, '__version__', '?'), '| torch', torch.__version__, '| pyg', torch_geometric.__version__)

In [ ]:
# helper unit tests (pure torch/numpy; fast)
!python -m pytest -q tests/test_nnsight_lab_helpers.py 2>&1 | tail -5

In [ ]:
# Load data windows, examples, graph cache, and the Lab
import numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, '.')
from src.training.dataset_v2 import load_sources, build_examples_v2
from src.training.train_v2 import GraphCache
from src.training.embeddings import primary_label
from src.explainability.nnsight_lab import Lab, HOOKS
from src.explainability import viz
from src.scoring.attack_stage import STAGE_NAMES

CKPT = 'data/processed/netjepa_weights.pt'
lab = Lab.from_checkpoint(CKPT)
L, H = lab.L, int(lab.meta.get('horizon') or 3)
sources = load_sources(include=('csv',))
train, val, _ = build_examples_v2(sources, L, H, 0.2, return_dropped=True)
cache = GraphCache(sources)
print(f'context_len={L} horizon={H} temperature={lab.T:.3f} | train={len(train)} val={len(val)} | SAE={lab.sae is not None} lens={lab.translators is not None} x_ref={lab.x_ref is not None}')
print('trained metrics:', json.dumps((lab.meta.get('validation_metrics') or {}).get('netjepa', {}), default=str)[:400])

In [ ]:
# 5. SELF-CHECK. All three diffs must be ~0 (< 1e-4) before trusting anything below.
chk = lab.selfcheck(cache, val)
print(json.dumps(chk, indent=1))
assert chk['ok'], 'nnsight path disagrees with the torch hooks - stop and send me this output'

In [ ]:
# Build an evaluation set: up to 1500 attack + 1500 benign validation windows (balanced so the plots are not all benign)
rng = np.random.default_rng(0)
att = [e for e in val if e.infil[0] == 1]; ben = [e for e in val if e.infil[0] == 0]
sel = [att[i] for i in rng.choice(len(att), min(1500, len(att)), replace=False)] + [ben[i] for i in rng.choice(len(ben), min(1500, len(ben)), replace=False)]
t0 = time.time(); C = lab.collect(cache, sel); print(f'collected {len(sel)} windows in {time.time()-t0:.0f}s')
labels = np.array([primary_label(e.context[-1]) for e in sel])
stage_pred = C['stage_now'].argmax(1)
thr = lab.meta.get('decision_threshold') or 0.5
print('families:', dict(zip(*np.unique(labels, return_counts=True))))
print(f'val accuracy @thr={thr:.2f}: {((C["p_cur"]>=thr)==(C["y"]==1)).mean():.3f}')
FIG = f'{DRIVE}/outputs/figures'; os.makedirs(FIG, exist_ok=True)
def show(fig, name):
    fig.savefig(f'{FIG}/{name}.png', dpi=130, bbox_inches='tight'); plt.show()

## A. Embedding space

In [ ]:
show(viz.embedding_overview(C['z'], labels, C['stage'], C['p_cur'], 'pca'), 'A1_embedding_pca')
show(viz.embedding_overview(C['z'], labels, C['stage'], C['p_cur'], 'tsne'), 'A2_embedding_tsne')

In [ ]:
viz.embedding_plotly(C['z'], labels, C['p_cur']).show()   # hover to see label / score

## B. Timeline of one capture (contiguous validation windows)

In [ ]:
# pick the validation file with the most attacks, take its first 500 windows in time order
from collections import Counter
fname = Counter(e.file for e in val if e.infil[0] == 1).most_common(1)[0][0]
seq = [e for e in val if e.file == fname][:500]
T = lab.collect(cache, seq)
show(viz.capture_timeline(T['p_cur'], T['p_fut'], [primary_label(e.context[-1]) for e in seq], T['stage_now'].argmax(1), thr), 'B1_timeline')
print(fname)

## C. Layers: where does the decision form?

In [ ]:
if lab.translators is not None:
    lens = lab.lens_trajectory(C['acts'])
    show(viz.lens_trajectory_plot(lens, labels), 'C1_lens_trajectory')
show(viz.cka_heatmap(C['acts']), 'C2_cka')
probes = json.load(open('data/processed/interpretability/probes.json'))
show(viz.probe_heatmap(probes), 'C3_probes')

## D. Causal activation patching (nnsight)
For flagged windows we copy activations from a **benign reference** (same graph structure) into one layer subspace at a time. If the flagged score collapses toward the benign score, that subspace *causally* carries the detection. Whole-layer patching is trivially 1.0 in a serial net, so each layer is split into 8 subspaces.

In [ ]:
flagged = [e for e, p in zip(sel, C['p_cur']) if e.infil[0] == 1 and p >= thr][:48]
print(len(flagged), 'flagged attack windows')
t0 = time.time(); grid = lab.patching_grid(cache, flagged); print(f'{time.time()-t0:.0f}s')
show(viz.patching_heatmap(grid), 'D1_patching')
show(viz.node_type_effects(grid), 'D2_node_types')
print({k: v for k, v in grid.items() if k in ('mean_p_flagged', 'mean_p_clean', 'n_valid_cur', 'n_valid_fut')})

In [ ]:
# Per-family patching: do different attacks route through different layers?
fams = [f for f, n in Counter(labels[(C['y'] == 1) & (C['p_cur'] >= thr)]).most_common(4)]
fig, ax = plt.subplots(1, len(fams), figsize=(4.2*len(fams), 4), sharey=True)
ax = np.atleast_1d(ax)
for a, f in zip(ax, fams):
    ex = [e for e, l, p in zip(sel, labels, C['p_cur']) if l == f and p >= thr][:32]
    g = lab.patching_grid(cache, ex)
    a.imshow(np.nan_to_num(g['cur']), vmin=0, vmax=1, cmap='inferno', aspect='auto'); a.set_title(f'{f} (n={len(ex)})', fontsize=9); a.grid(False)
    a.set_yticks(range(len(g['points']))); a.set_yticklabels([viz.LAYER_NAMES.get(p, p) for p in g['points']], fontsize=7); a.set_xlabel('subspace')
fig.suptitle('causal effect on current score, per attack family'); fig.tight_layout(); show(fig, 'D3_patching_by_family')

## E. Embedding dimensions (nnsight mean-ablation)

In [ ]:
sub = flagged[:48]
abl = lab.dim_ablation(cache, sub, z_mean=C['z'][C['y'] == 0].mean(0))
zl = {l: C['z'][labels == l].mean(0) for l in sorted(set(labels), key=lambda s: (s != 'BENIGN', s)) if (labels == l).sum() >= 10}
show(viz.dim_importance(abl, zl), 'E1_dim_importance')
print('dims needed for 90% of total importance:', int((np.cumsum(np.sort(abl['cur'])[::-1]) / abl['cur'].sum() < 0.9).sum()) + 1, 'of', len(abl['cur']))

## F. Sparse autoencoder: are the features real? (nnsight)
* **Overview** - dead features, class specificity, how monosemantic the labels are.
* **Ablation** - remove one feature's contribution from z_t: if P(attack) drops, the feature is *causally used*.
* **Steering** - add a feature's direction to *benign* windows: does P(attack) rise?
* **Fidelity** - replace z_t by its SAE reconstruction: do decisions survive (vs destructive baselines)?

In [ ]:
if lab.sae is not None:
    labels_json = lab.sae_labels
    fig, Hs = viz.sae_overview(lab.sae, C['z'], labels, labels_json); show(fig, 'F1_sae_overview')
    ridx = np.random.default_rng(1).choice(len(labels), 400, replace=False); ridx = ridx[np.argsort(labels[ridx], kind='stable')]  # 400 random windows grouped by family
    show(viz.sae_raster(Hs[ridx], labels[ridx], 40, 400), 'F2_sae_raster')
    show(viz.sae_fidelity_bar(lab.sae_fidelity(cache, sel[:256] + sel[-256:])), 'F3_sae_fidelity')

In [ ]:
if lab.sae is not None:
    eff = lab.sae_effects(cache, flagged, top_k=12)
    show(viz.sae_effects_bar(eff, 'ablating SAE features on FLAGGED windows'), 'F4_sae_ablation')
    # steering with the most attack-specific, frequently-firing labelled features
    cand = sorted(((v['activation_frequency'], int(k)) for k, v in lab.sae_labels.items() if v['dominant_label'] != 'BENIGN' and v['purity'] >= 0.7), reverse=True)[:5]
    feats = [f for _, f in cand]
    print('steering features:', [(f, lab.sae_labels[str(f)]['label']) for f in feats])
    if feats:
        show(viz.steering_curves(lab.sae_steering(cache, [e for e in sel if e.infil[0] == 0][:64], feats)), 'F5_sae_steering')

## G. One window

In [ ]:
# pick the most confidently flagged attack window
i = int(np.argmax(np.where(C['y'] == 1, C['p_cur'], -1))); ex = sel[i]
print('label:', labels[i], '| P(attack)=%.3f' % C['p_cur'][i], '| forecast=%.3f' % C['p_fut'][i], '| stage:', STAGE_NAMES[stage_pred[i]])
show(viz.attribution_bar(lab.grad_x_input(cache, ex)), 'G1_attribution')
show(viz.rollout_plot(lab.rollout_bands(C['z'][i], steps=H + 2), C['p_cur'][i]), 'G2_rollout')
if lab.sae is not None:
    show(viz.sae_effects_bar(lab.sae_effects(cache, [ex], top_k=10), 'SAE features of THIS window'), 'G3_window_sae')

## H. Model quality

In [ ]:
show(viz.calibration_plot(C['p_cur'], C['y']), 'H1_calibration')
show(viz.confusion_plot(C['stage'], stage_pred), 'H2_stage_confusion')

In [ ]:
print('figures saved to', FIG); print(sorted(os.listdir(FIG)))

## How to read the results (and what NOT to conclude)
* **Patching effect ~1 in one subspace and ~0 elsewhere** = a localised causal mechanism; **spread evenly** = distributed. Predictor layers should only matter for the forecast panel.
* **SAE ablation near 0 for a labelled feature** means the label is descriptive but the feature is not used by the heads - do not present it as an 'attack detector'.
* **SAE fidelity**: if reconstruction agreement is far below ~0.95 the SAE misses decision-relevant structure and all SAE conclusions are weak.
* The graphs in CSV mode have one network node and a dummy host (no real IPs), so the node-type panel is about window-level vs placeholder host nodes, not real machines. Real host-level analysis needs PCAP windows.
* Everything is evidence about *this checkpoint on CIC-IDS-2017*, not a general mechanism.